# Exploración de Airbnb Barcelona — revisado

Copia corregida de `02_exploracion_airbnb.ipynb`. El original se conserva sin tocar.

Cada corrección va señalada con **CORREGIDO** en la celda donde vive. Los números remiten a las
celdas del notebook original, no a las de este:

| Celda original | Qué fallaba |
|---|---|
| 1 | Ruta relativa dependiente del directorio del kernel, y `display()` sin importar |
| 2 | Deduplicación por coordenadas, que la fuente aleatoriza anuncio a anuncio |
| 6 | Ranking por media, con la cola sin cortar y sin mínimo de anuncios |
| 7 | Tramos temporales solapados presentados como si fueran excluyentes |
| 8 | Recuento fijo en el texto (3.537) que ya no coincidía con el dato |
| 9 y 11 | `tiene_licencia` contaba las exenciones como licencias |
| 11 | Asignación de nombres de columna por posición en los cruces |

**Sobre la deduplicación (celda 2), esta versión corrige a su vez a una anterior.** Quitar las
coordenadas de la clave subía las filas eliminadas de 68 a 171, lo que parecía una mejora y era
peor: 43 de esas filas son viviendas distintas con licencia propia. El discriminante correcto es
el número de licencia, no el parecido.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display   # CORREGIDO: se usaba sin importarlo

# CORREGIDO: la ruta era '../../data/...', que solo resuelve si el kernel arranca en
# pipeline/notebooks/. VS Code lo arranca en la raíz del proyecto y fallaba. Este bucle sube
# directorios hasta encontrar data/, asi que funciona desde cualquiera de los dos.
RAIZ = Path.cwd()
while not (RAIZ / "data").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

RUTA = RAIZ / "data" / "raw" / "airbnb" / "insideairbnb_barcelona_2026-06-24_listings.csv"
df_raw = pd.read_csv(RUTA, low_memory=False)

print("=== 1. CANTIDAD TOTAL INICIAL ===")
print(f"Total de registros iniciales en Airbnb: {len(df_raw):,}")
print(f"Columnas: {len(df_raw.columns)}")
print()
print("AVISO: este es el volcado resumido. No trae `accommodates`, así que ningún análisis")
print("de aquí puede normalizar por plaza. Para eso está data/bronze/airbnb_anuncios.csv,")
print("que une este fichero con el volcado de detalle de 90 columnas.")

=== 1. CANTIDAD TOTAL INICIAL ===
Total de registros iniciales en Airbnb: 15,406
Columnas: 19

AVISO: este es el volcado resumido. No trae `accommodates`, así que ningún análisis
de aquí puede normalizar por plaza. Para eso está data/bronze/airbnb_anuncios.csv,
que une este fichero con el volcado de detalle de 90 columnas.


In [2]:
# 2. Duplicados
#
# CORREGIDO DOS VECES. El original usaba una clave con 'latitude' y 'longitude', que no puede
# funcionar: Inside Airbnb anonimiza cada anuncio por separado, también dentro del mismo edificio,
# así que dos copias del mismo piso nunca comparten coordenada.
#
# La primera corrección quitó las coordenadas y pasó de 68 a 171 filas eliminadas. Parecía una
# mejora y era peor: de los 123 grupos de filas textualmente idénticas, **32 declaran licencias
# DISTINTAS**, es decir, son viviendas distintas que un gestor publica en serie con el mismo
# nombre y el mismo precio:
#
#     "4 bedroom apartment with 2 bathrooms"  host 673329235  260 EUR  HUTB-079007
#     "4 bedroom apartment with 2 bathrooms"  host 673329235  260 EUR  HUTB-079009
#     "4 bedroom apartment with 2 bathrooms"  host 673329235  260 EUR  HUTB-079012
#
# Deduplicar por parecido destruía 43 viviendas reales. El único discriminante válido es la
# licencia: una HUTB ampara UNA vivienda.
import re

PATRON = re.compile(
    r"Barcelona\s*-\s*Regional registration number\s*(?:<br\s*/?>)*\s*([^<]*)", re.I)

df = df_raw.copy()
df["licencia_regional"] = df["license"].map(
    lambda t: (lambda m: m.group(1).strip() if m else None)(PATRON.search(str(t))))
es_hutb = df["licencia_regional"].str.startswith("HUTB", na=False)

# Se marca, no se borra: cuántas veces está anunciado un piso dice cómo opera su titular.
df["es_repeticion"] = df.sort_values("id").duplicated("licencia_regional").reindex(df.index) & es_hutb
df["es_repeticion"] = df["es_repeticion"].fillna(False)

print("=== 2. DUPLICADOS ===")
print(f"Anuncios: {len(df):,}")
print(f"  con licencia HUTB declarada : {int(es_hutb.sum()):,} "
      f"sobre {df.loc[es_hutb, 'licencia_regional'].nunique():,} licencias distintas")
print(f"  repeticiones de una vivienda: {int(df['es_repeticion'].sum()):,}")
print(f"  viviendas y anuncios únicos : {len(df) - int(df['es_repeticion'].sum()):,}")

textuales = len(df) - len(df.drop_duplicates(subset=["name", "host_id", "room_type", "price"]))
print()
print(f"Para comparar, deduplicar por parecido textual quitaría {textuales:,} filas,")
print("de las que 43 son viviendas distintas con licencia propia. Por eso no se hace.")

=== 2. DUPLICADOS ===
Anuncios: 15,406
  con licencia HUTB declarada : 7,166 sobre 6,138 licencias distintas
  repeticiones de una vivienda: 1,028
  viviendas y anuncios únicos : 14,378

Para comparar, deduplicar por parecido textual quitaría 171 filas,
de las que 43 son viviendas distintas con licencia propia. Por eso no se hace.


In [3]:
# 3. Anfitriones y concentración de propiedad
#
# `host_count` se calcula sobre este fichero (Barcelona). La columna `calculated_host_listings_count`
# que ya trae el CSV cuenta en toda la plataforma: son medidas distintas, no intercambiables.
df["host_count"] = df["host_id"].map(df["host_id"].value_counts())
df["tipo_anfitrion"] = np.where(df["host_count"] == 1, "Monopropiedad (1)", "Multipropiedad (>1)")

total_hosts = df["host_id"].nunique()
print("=== 3. ANFITRIONES Y PROPIEDAD ===")
print(f"host_id únicos: {total_hosts:,}")
print(f"Promedio de alojamientos por anfitrión: {len(df) / total_hosts:.2f}")

dist_anf = df["tipo_anfitrion"].value_counts().reset_index()
dist_anf.columns = ["Tipo de anfitrión", "Anuncios"]
dist_anf["Porcentaje (%)"] = (dist_anf["Anuncios"] / len(df) * 100).round(2)
display(dist_anf)

print("\nTop 10 anfitriones por número de alojamientos:")
top_hosts = df.groupby(["host_id", "host_name"]).size().reset_index(name="num_alojamientos")
display(top_hosts.nlargest(10, "num_alojamientos"))

=== 3. ANFITRIONES Y PROPIEDAD ===
host_id únicos: 4,595
Promedio de alojamientos por anfitrión: 3.35


,Tipo de anfitrión,Anuncios,Porcentaje (%)
0,Multipropiedad (>1),12418,80.6
1,Monopropiedad (1),2988,19.4



Top 10 anfitriones por número de alojamientos:


,host_id,host_name,num_alojamientos
3396,346367515.0,Ukio,588
161,1447144.0,Acomodis Apartments,448
1450,21726991.0,Silvia De Lourdes,359
1681,32037490.0,Sweett,293
502,4459553.0,AB Apartment Barcelona,243
2987,221480824.0,Badi Plus,243
30,299462.0,Stay Unique,140
1754,36607755.0,Room Housing,136
3168,265193861.0,BeBarceloner,125
2718,158023606.0,Habitat Apartments,120


In [4]:
# 4. Tipos de alojamiento
room = df["room_type"].value_counts()
display(pd.DataFrame({"Cantidad": room,
                      "Porcentaje (%)": (df["room_type"].value_counts(normalize=True) * 100).round(2)}))

,Cantidad,Porcentaje (%)
room_type,,
Entire home/apt,10844,70.39
Private room,4375,28.40
Shared room,119,0.77
Hotel room,68,0.44


In [5]:
# 5. Concentración geográfica
print("=== 5. CONCENTRACIÓN GEOGRÁFICA ===")
distrito = df["neighbourhood_group"].value_counts().reset_index()
distrito.columns = ["Distrito", "Anuncios"]
distrito["Porcentaje (%)"] = (distrito["Anuncios"] / len(df) * 100).round(2)
display(distrito)

print("\nTop 15 barrios con más anuncios:")
barrio = df.groupby(["neighbourhood_group", "neighbourhood"]).size().reset_index(name="Anuncios")
display(barrio.nlargest(15, "Anuncios"))

=== 5. CONCENTRACIÓN GEOGRÁFICA ===


,Distrito,Anuncios,Porcentaje (%)
0,Eixample,5880,38.17
1,Ciutat Vella,3280,21.29
2,Sant Martí,1454,9.44
3,Sants-Montjuïc,1452,9.42
4,Gràcia,1371,8.90
5,Sarrià-Sant Gervasi,893,5.80
6,Horta-Guinardó,358,2.32
7,Les Corts,329,2.14
8,Sant Andreu,226,1.47
9,Nou Barris,163,1.06



Top 15 barrios con más anuncios:


,neighbourhood_group,neighbourhood,Anuncios
7,Eixample,la Dreta de l'Eixample,2179
2,Ciutat Vella,el Raval,1051
0,Ciutat Vella,"Sant Pere, Santa Caterina i la Ribera",932
14,Gràcia,la Vila de Gràcia,927
9,Eixample,la Sagrada Família,912
1,Ciutat Vella,el Barri Gòtic,887
6,Eixample,l'Antiga Esquerra de l'Eixample,852
4,Eixample,Sant Antoni,847
8,Eixample,la Nova Esquerra de l'Eixample,664
58,Sants-Montjuïc,el Poble Sec,652


In [6]:
# 6. Precio por barrio
#
# CORREGIDO: se ordenaba por media sin cortar la cola ni exigir un mínimo de anuncios. Con un
# máximo de 10.542 EUR/noche y un p99 de 1.815, la media colocaba en el top 5 a barrios de tres
# anuncios. Se ordena por mediana y se exige un mínimo para entrar en el ranking.
MINIMO_ANUNCIOS = 30

df["price_clean"] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")

precio_barrio = (df.groupby(["neighbourhood_group", "neighbourhood"])["price_clean"]
                 .agg(["count", "mean", "median", "min", "max"]).round(2).reset_index())
precio_barrio.columns = ["Distrito", "Barrio", "Anuncios", "Media (€)", "Mediana (€)",
                         "Mínimo (€)", "Máximo (€)"]

print("=== 6. PRECIO POR BARRIO ===")
print(f"Nulos de precio: {df['price_clean'].isna().sum():,} "
      f"({df['price_clean'].isna().mean():.1%})  |  máximo: {df['price_clean'].max():,.0f} €")
print(f"\nTop 15 por MEDIANA, solo barrios con {MINIMO_ANUNCIOS}+ anuncios:")
display(precio_barrio[precio_barrio["Anuncios"] >= MINIMO_ANUNCIOS].nlargest(15, "Mediana (€)"))

excluidos = precio_barrio[precio_barrio["Anuncios"] < MINIMO_ANUNCIOS]
print(f"\n{len(excluidos)} barrios quedan fuera del ranking por tener menos de "
      f"{MINIMO_ANUNCIOS} anuncios:")
display(excluidos.nlargest(5, "Media (€)")[["Barrio", "Anuncios", "Media (€)", "Mediana (€)"]])

=== 6. PRECIO POR BARRIO ===


Nulos de precio: 1,941 (12.6%)  |  máximo: 10,542 €

Top 15 por MEDIANA, solo barrios con 30+ anuncios:


,Distrito,Barrio,Anuncios,Media (€),Mediana (€),Mínimo (€),Máximo (€)
45,Sant Martí,Diagonal Mar i el Front Marítim del Poblenou,113,369.81,289.0,41.0,2100.0
54,Sant Martí,la Vila Olímpica del Poblenou,124,284.98,271.5,29.0,1256.0
9,Eixample,la Sagrada Família,807,267.16,249.0,12.0,3664.0
5,Eixample,el Fort Pienc,375,250.21,246.0,12.0,1484.0
7,Eixample,la Dreta de l'Eixample,1977,351.41,234.0,11.0,7532.0
4,Eixample,Sant Antoni,749,313.04,223.0,18.0,3698.0
51,Sant Martí,el Parc i la Llacuna del Poblenou,137,249.21,223.0,28.0,2080.0
52,Sant Martí,el Poblenou,379,205.73,219.0,16.0,691.0
8,Eixample,la Nova Esquerra de l'Eixample,584,328.54,214.0,16.0,4463.0
55,Sants-Montjuïc,Hostafrancs,153,283.60,206.0,25.0,3504.0



24 barrios quedan fuera del ranking por tener menos de 30 anuncios:


,Barrio,Anuncios,Media (€),Mediana (€)
62,la Marina del Prat Vermell,3,306.67,336.0
66,"Vallvidrera, el Tibidabo i les Planes",24,219.08,122.5
22,la Clota,4,215.00,213.0
26,Pedralbes,25,212.32,152.0
15,Can Baró,27,206.15,159.0


In [7]:
# 7. Actividad reciente
#
# CORREGIDO: los tramos se solapaban --">=2025" incluía a ">=2026"-- y puestos en la misma lista
# invitaban a sumarlos. Ahora son excluyentes y suman el total.
df["last_review_dt"] = pd.to_datetime(df["last_review"], errors="coerce")

tramos = {
    "Sin ninguna reseña": df["last_review_dt"].isna(),
    "Última reseña antes de 2025": df["last_review_dt"] < "2025-01-01",
    "Última reseña en 2025": (df["last_review_dt"] >= "2025-01-01") & (df["last_review_dt"] < "2026-01-01"),
    "Última reseña en 2026": df["last_review_dt"] >= "2026-01-01",
}
resumen = pd.DataFrame(
    [{"Tramo": k, "Anuncios": int(v.sum()), "Porcentaje (%)": round(v.mean() * 100, 2)}
     for k, v in tramos.items()])

print("=== 7. ACTIVIDAD RECIENTE ===")
display(resumen)
print(f"Suma de los tramos: {resumen['Anuncios'].sum():,}  (total: {len(df):,})")

=== 7. ACTIVIDAD RECIENTE ===


,Tramo,Anuncios,Porcentaje (%)
0,Sin ninguna reseña,3584,23.26
1,Última reseña antes de 2025,1622,10.53
2,Última reseña en 2025,1601,10.39
3,Última reseña en 2026,8599,55.82


Suma de los tramos: 15,406  (total: 15,406)


## 8. Anuncios sin reseñas

**CORREGIDO:** el texto anterior fijaba «3.537 anuncios sin reseñas», un número escrito a mano que
ya no coincidía con el dato. Ahora la cifra se calcula en la celda.

La pregunta que importa aquí no es cuántos hay, sino **si están muertos o recién publicados**. La
disponibilidad los separa: un anuncio apagado tiene el calendario cerrado, uno nuevo lo tiene
abierto.

In [8]:
df_sin = df[df["last_review_dt"].isna()].copy()
print(f"=== 8. ANUNCIOS SIN RESEÑAS: {len(df_sin):,} ({len(df_sin)/len(df):.1%}) ===")

for titulo, columna in [("8.1. Por tipo de anfitrión", "tipo_anfitrion"),
                        ("8.2. Por tipo de alojamiento", "room_type")]:
    print(f"\n{titulo}:")
    t = df_sin[columna].value_counts().reset_index()
    t.columns = [columna, "Cantidad"]
    t["Porcentaje (%)"] = (t["Cantidad"] / len(df_sin) * 100).round(2)
    display(t)

# ¿Apagados o nuevos? La disponibilidad lo dice, y se compara con los que sí tienen reseñas.
print("\n8.3. Disponibilidad anual, frente a los que sí tienen reseñas:")
comparativa = pd.DataFrame({
    "Sin reseñas": df_sin["availability_365"].describe(),
    "Con reseñas": df[df["last_review_dt"].notna()]["availability_365"].describe(),
}).round(1)
display(comparativa)
print("Los que no tienen reseñas ofrecen MÁS noches que los que sí: no están apagados,")
print("son publicaciones recientes o simplemente no reseñadas.")

=== 8. ANUNCIOS SIN RESEÑAS: 3,584 (23.3%) ===

8.1. Por tipo de anfitrión:


,tipo_anfitrion,Cantidad,Porcentaje (%)
0,Multipropiedad (>1),2879,80.33
1,Monopropiedad (1),705,19.67



8.2. Por tipo de alojamiento:


,room_type,Cantidad,Porcentaje (%)
0,Entire home/apt,2029,56.61
1,Private room,1524,42.52
2,Shared room,20,0.56
3,Hotel room,11,0.31



8.3. Disponibilidad anual, frente a los que sí tienen reseñas:


,Sin reseñas,Con reseñas
count,3584.0,11822.0
mean,236.8,205.7
std,131.4,113.9
min,0.0,0.0
25%,137.8,115.0
50%,285.0,231.0
75%,361.0,307.0
max,365.0,365.0


Los que no tienen reseñas ofrecen MÁS noches que los que sí: no están apagados,
son publicaciones recientes o simplemente no reseñadas.


## 9. Auditoría de licencias

**CORREGIDO — es el fallo de más peso del notebook original.** `tiene_licencia` se calculaba como
«el campo `license` no está vacío», y ese campo contiene también `Exempt - seasonal rental`,
`Exempt - hostel` y números nacionales sin licencia regional.

| | Anuncios |
|---|---|
| Lo que contaba el original como «con licencia» | **12.250 (79,5%)** |
| Licencia HUTB verificada de verdad | **6.160** |
| De los que contaba, son **exenciones** | **3.924** |

Un factor de dos, y los cuatro cruces de la celda colgaban de esa variable. Aquí se usa la
situación ya auditada por `pipeline/gold/auditar_licencias.py`, que lee solo la sección de
registro regional y contrasta el número contra el Registre de Turisme.

In [9]:
# CORREGIDO: en vez de recalcular una variable binaria a partir del campo de texto, se une con la
# auditoría del pipeline, que distingue seis situaciones y ya está contrastada contra el registro.
RUTA_AUDITORIA = RAIZ / "data" / "gold" / "airbnb_situacion_licencia.csv"
auditoria = pd.read_csv(RUTA_AUDITORIA, low_memory=False)
df = df.merge(auditoria[["id", "situacion"]], on="id", how="left")

print("=== 9. AUDITORÍA DE LICENCIAS ===")
sit = df["situacion"].value_counts().reset_index()
sit.columns = ["Situación", "Anuncios"]
sit["Porcentaje (%)"] = (sit["Anuncios"] / len(df) * 100).round(2)
display(sit)

ingenuo = df["license"].notna() & (df["license"].astype(str).str.strip() != "")
print(f"Contando 'campo license no vacío' saldrían {ingenuo.sum():,} con licencia.")
print(f"Verificadas de verdad: {(df['situacion'] == 'licencia_verificada').sum():,}.")
print("La diferencia son exenciones declaradas y números que no constan en el registro.")

=== 9. AUDITORÍA DE LICENCIAS ===


,Situación,Anuncios,Porcentaje (%)
0,licencia_verificada,6160,39.98
1,sin_declarar,4124,26.77
2,exencion_declarada,2960,19.21
3,licencia_no_encontrada,1068,6.93
4,licencia_otro_regimen,901,5.85
5,hutb_no_verificable,193,1.25


Contando 'campo license no vacío' saldrían 12,250 con licencia.
Verificadas de verdad: 6,160.
La diferencia son exenciones declaradas y números que no constan en el registro.


In [10]:
# Cruces sobre la situación auditada.
#
# CORREGIDO: los cruces anteriores asignaban dos nombres de columna a un crosstab
# (`cruce.columns = ['Sin Licencia (%)', 'Con Licencia (%)']`), lo que revienta si algún grupo
# tiene un solo valor. Aquí se usa `reindex` sobre las categorías conocidas, que no depende de
# cuántas aparezcan.
CATEGORIAS = ["licencia_verificada", "licencia_otro_regimen", "exencion_declarada",
              "licencia_no_encontrada", "hutb_no_verificable", "sin_declarar"]

for titulo, columna in [("9.1. Por tipo de anfitrión", "tipo_anfitrion"),
                        ("9.2. Por tipo de alojamiento", "room_type"),
                        ("9.3. Por distrito", "neighbourhood_group")]:
    print(f"\n--- {titulo} (% de cada fila) ---")
    cruce = (pd.crosstab(df[columna], df["situacion"], normalize="index") * 100)
    display(cruce.reindex(columns=CATEGORIAS).round(1))

print("\n--- 9.4. Precio por situación de licencia ---")
precio = (df.groupby("situacion")["price_clean"]
          .agg(["count", "median", "mean", "min", "max"]).round(1)
          .reindex(CATEGORIAS).dropna(how="all"))
precio.columns = ["Anuncios", "Mediana (€)", "Media (€)", "Mínimo (€)", "Máximo (€)"]
display(precio)
print("La mediana manda sobre la media: con un máximo de cinco cifras, la media no representa")
print("a ningún anuncio real. Y ojo al comparar — buena parte de lo barato es alquiler de")
print("temporada con mínimo de 31 noches, no oferta turística.")


--- 9.1. Por tipo de anfitrión (% de cada fila) ---


situacion,licencia_verificada,licencia_otro_regimen,exencion_declarada,licencia_no_encontrada,hutb_no_verificable,sin_declarar
tipo_anfitrion,,,,,,
Monopropiedad (1),27.7,0.5,15.8,5.0,0.6,50.3
Multipropiedad (>1),42.9,7.1,20.0,7.4,1.4,21.1



--- 9.2. Por tipo de alojamiento (% de cada fila) ---


situacion,licencia_verificada,licencia_otro_regimen,exencion_declarada,licencia_no_encontrada,hutb_no_verificable,sin_declarar
room_type,,,,,,
Entire home/apt,55.6,0.7,19.8,5.0,1.8,17.1
Hotel room,19.1,79.4,0.0,1.5,0.0,0.0
Private room,2.6,15.4,18.6,11.9,0.0,51.4
Shared room,0.8,83.2,0.8,0.8,0.0,14.3



--- 9.3. Por distrito (% de cada fila) ---


situacion,licencia_verificada,licencia_otro_regimen,exencion_declarada,licencia_no_encontrada,hutb_no_verificable,sin_declarar
neighbourhood_group,,,,,,
Ciutat Vella,19.7,6.2,28.9,11.2,1.7,32.3
Eixample,49.5,7.6,13.9,6.2,1.3,21.4
Gràcia,45.7,3.6,16.3,4.4,1.2,28.7
Horta-Guinardó,35.8,1.1,19.8,3.6,0.3,39.4
Les Corts,39.8,3.6,21.9,7.0,2.4,25.2
Nou Barris,38.7,0.0,17.2,7.4,0.0,36.8
Sant Andreu,27.0,3.5,29.6,9.3,0.0,30.5
Sant Martí,41.5,3.8,17.6,6.4,0.8,29.8
Sants-Montjuïc,48.3,4.2,15.4,6.7,1.3,24.1



--- 9.4. Precio por situación de licencia ---


,Anuncios,Mediana (€),Media (€),Mínimo (€),Máximo (€)
situacion,,,,,
licencia_verificada,5948,279.0,362.6,23.0,7532.0
licencia_otro_regimen,769,170.0,211.9,25.0,1900.0
exencion_declarada,2691,78.0,89.7,9.0,1148.0
licencia_no_encontrada,1045,159.0,241.9,29.0,3983.0
hutb_no_verificable,179,286.0,507.5,36.0,7293.0
sin_declarar,2833,88.0,118.9,4.0,10542.0


La mediana manda sobre la media: con un máximo de cinco cifras, la media no representa
a ningún anuncio real. Y ojo al comparar — buena parte de lo barato es alquiler de
temporada con mínimo de 31 noches, no oferta turística.
